# Modul 06 · nb06 — Edge Deploy di Jetson Orin: **TensorRT Edge-LLM**

> **Domain sertifikasi (NCA-GENL): Software Development (24%) — *model deployment & serving* (edge).**
> ⚠️ **Lab ini TIDAK dijalankan di Colab.** Ia berjalan di **NVIDIA Jetson Orin** (perangkat edge nyata). Notebook ini adalah **runbook**: tiap sel adalah perintah yang dijalankan di Orin lewat terminal/SSH. Output di bawah tiap bagian adalah **hasil sungguhan** dari run yang sudah diverifikasi di Jetson Orin Nano Super 8 GB (JetPack 6.2, CUDA 12.6, TensorRT 10.3).

## Dari hosted NIM ke edge NVIDIA-native

Di **nb02** kita memakai NIM *hosted* (model jalan di server NVIDIA, kita cukup memanggil `/v1`). Itu sempurna untuk belajar & prototipe, tapi punya batas: butuh internet, data keluar ke cloud, dan ada biaya/rate-limit. Untuk **produksi edge** — robot, kamera, kiosk, perangkat lapangan — kita ingin model **berjalan di perangkat itu sendiri**: offline, latency rendah, data tak ke mana-mana.

Di **Modul 04** kamu mem-*fine-tune* sebuah spesialis kecil — **`qwen3-0.6b-spesialis`** (Qwen3-0.6B + QLoRA, ekstraktor pesanan→JSON). Di nb06 ini kita **deploy spesialis itu secara NVIDIA-native di Jetson Orin** memakai **TensorRT Edge-LLM**: runtime C++ performa tinggi NVIDIA untuk LLM di perangkat embedded (Orin → Thor). Alurnya: **checkpoint HuggingFace → ONNX → engine TensorRT → inference C++** (tanpa interpreter Python di jalur inference).

Inilah penutup tema **"satu klien, banyak backend"**: kontrak yang sama, kali ini backend-nya adalah **engine TensorRT yang dioptimalkan untuk GPU Orin**.

## Kenapa engine TensorRT **wajib** di-build di perangkat target?

Pertanyaan wajar: *"kenapa tidak build di Colab/x86 saja lalu copy file-nya ke Jetson?"* Jawabannya memisahkan **dua "build" yang portabilitasnya berbeda**:

| Tahap | Artefak | Portabel? | Di mana |
|-------|---------|-----------|---------|
| **Export ONNX** | `model.onnx` + bobot | ✅ **Ya** — graph + bobot, tak terikat hardware | Colab / x86 / venv-CPU (di mana saja) |
| **Build engine** | `llm.engine` (TensorRT) | ❌ **Tidak** — terikat arsitektur GPU + versi TensorRT | **Wajib di GPU target** |

Sebuah **engine TensorRT** meng-embed **machine code (SASS) + pilihan taktik** yang disetel untuk **compute-capability GPU spesifik** dan **versi TensorRT spesifik**. Contoh konkret: Colab T4 = **sm_75 (Turing)**; Orin Nano = **sm_87 (Ampere)**. Engine yang dibangun di T4 akan **ditolak saat di-load** di Orin (mismatch arch/versi) — bukan sekadar lambat, tapi **tidak jalan**.

**Maka pembagiannya:** export ONNX boleh di mana saja (kita lakukan di venv-CPU di Orin agar self-contained, tapi Colab pun bisa) → **build engine + inference WAJIB di Orin**. Binary runtime (`llm_build`/`llm_inference`) juga arch-spesifik (aarch64 + kernel sm_87) sehingga di-*compile* untuk target `jetson-orin`. Itulah alasan teknis "compile di tempat deploy".

## Prasyarat (sudah ada di Orin ini)

- **Jetson Orin Nano 8 GB**, JetPack **6.2** (L4T R36.4), **CUDA 12.6**, **TensorRT 10.3**, `cmake` + `g++-11`, Docker.
- Ruang disk: simpan kerja di volume lega (di sini `/media/jetson/DATA`, ~30 GB free) — root fs Jetson biasanya kecil.
- Model: **`chmdznr/qwen3-0.6b-spesialis`** (safetensors) sudah di HuggingFace (dari Modul 04 nb07 yang mengekspor **dua format**: GGUF untuk Ollama/llama.cpp **dan** safetensors untuk jalur TensorRT ini).

> Semua perintah dijalankan sebagai user `jetson`. Set workspace sekali:
> ```bash
> export W=/media/jetson/DATA/edgellm-work && mkdir -p $W && cd $W
> ```

## Part A — Export checkpoint → ONNX (portabel)

`tensorrt_edgellm` adalah frontend Python yang membaca safetensors HuggingFace dan mengemit **ONNX + sidecar** (tokenizer, chat-template, tabel embedding) yang dimakan engine builder C++.

### Gotcha #1 — torch di Jetson
- Tutorial resmi NVIDIA menjalankan export di **container PyTorch NVIDIA (~20 GB)**. Di Orin Nano root-disk kecil ini **tak muat** (Docker 29 menyimpan layer container di **`/var/lib/containerd` pada root fs**, *bukan* di docker-root yang kita pindah ke DATA) → pull gagal *"no space left on device"*.
- torch **sistem** Jetson (2.5) **terlalu lama**: `torch.onnx.export()` belum punya argumen `dynamic_shapes` yang dibutuhkan repo → `TypeError`.
- **Solusi ramping & terbukti:** **venv terisolasi dengan `torch==2.12.0+cpu`**. Export = *tracing* graph di CPU, jadi torch CPU cukup — tanpa container, tanpa GPU, **tanpa mengganggu torch sistem** (yang dipakai project lain di Orin).

### Gotcha #2 — `huggingface_hub`
`transformers` 4.57 butuh `huggingface_hub < 1.0`. Kalau ter-upgrade ke 1.x, `import transformers` gagal. Pin `"huggingface_hub<1.0"`.

In [ ]:
# === Part A: export venv (torch CPU 2.12) + export ONNX ===  (di Orin)
export W=/media/jetson/DATA/edgellm-work && cd $W

# 1) Unduh model spesialis (safetensors) dari HuggingFace
pip3 install -q "huggingface_hub<1.0"
python3 -c "from huggingface_hub import snapshot_download as d; d('chmdznr/qwen3-0.6b-spesialis', local_dir='$W/qwen3-0.6b-spesialis')"

# 2) Clone TensorRT Edge-LLM + submodule (kecil: googletest/json/NVTX)
git clone --depth 1 https://github.com/NVIDIA/TensorRT-Edge-LLM.git $W/TensorRT-Edge-LLM
cd $W/TensorRT-Edge-LLM && git submodule update --init --recursive

# 3) venv export TERISOLASI: torch CPU 2.12 (punya dynamic_shapes) — tak menyentuh torch sistem
python3 -m venv $W/expvenv
$W/expvenv/bin/pip install -q "torch>=2.7" --index-url https://download.pytorch.org/whl/cpu
$W/expvenv/bin/pip install -q "transformers==4.57.6" onnx onnxscript onnx-graphsurgeon "numpy<2.3" "huggingface_hub<1.0"
$W/expvenv/bin/pip install -q --no-deps $W/TensorRT-Edge-LLM   # paket tensorrt_edgellm

# 4) Export → ONNX (CPU). Batasi thread agar santun ke beban lain.
cd $W
OMP_NUM_THREADS=2 $W/expvenv/bin/tensorrt-edgellm-export ./qwen3-0.6b-spesialis ./qwen3-onnx

**Output (terverifikasi):** export selesai ~2 menit, menghasilkan:
```
Export complete
  output dir: ./qwen3-onnx
  thinker : ./qwen3-onnx/llm/model.onnx  (3.6 MB) + embedding.safetensors (311.2 MB)
```
Isi `./qwen3-onnx/llm/`: `model.onnx` (graph), `model.onnx.data` (bobot eksternal), `embedding.safetensors`, plus sidecar tokenizer & `processed_chat_template.json`.

## Part B — Build C++ runtime di Orin

Sekarang bagian yang **wajib native**: kompilasi runtime C++ (`llm_build`, `llm_inference`, dst.) untuk arsitektur Orin (`aarch64` + kernel `sm_87`), me-link ke TensorRT 10.3 sistem.

### Gotcha #3 — environment CUDA
SSH non-login shell **tidak** me-load env CUDA JetPack, jadi `nvcc` tak ada di PATH dan cmake gagal: `nvcc fatal: Failed to preprocess host compiler properties`. **Export env CUDA dulu.**

### Gotcha #4 — target & RAM
- Pakai `-DEMBEDDED_TARGET=jetson-orin` + toolchain `aarch64` → menangani `sm_87` otomatis (default repo `80;86;89` **tak** termasuk Orin).
- Orin Nano 8 GB: `make -j4` (bukan `-j$(nproc)`) agar tidak OOM saat kompilasi.

In [ ]:
# === Part B: build runtime C++ (native, sm_87) ===  (di Orin)
# WAJIB: env CUDA (kalau tidak, nvcc gagal "preprocess host compiler properties")
export PATH=/usr/local/cuda/bin:$PATH
export CUDA_HOME=/usr/local/cuda
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib/aarch64-linux-gnu:$LD_LIBRARY_PATH

cd /media/jetson/DATA/edgellm-work/TensorRT-Edge-LLM
rm -rf build && mkdir build && cd build
cmake .. \
  -DCMAKE_BUILD_TYPE=Release \
  -DTRT_PACKAGE_DIR=/usr \
  -DCMAKE_TOOLCHAIN_FILE=cmake/aarch64_linux_toolchain.cmake \
  -DEMBEDDED_TARGET=jetson-orin
make -j4          # Orin Nano 8 GB: -j4, JANGAN -j$(nproc) (OOM). ~15-25 menit.

ls -la examples/llm/llm_build examples/llm/llm_inference   # binari + libNvInfer_edgellm_plugin.so

**Output (terverifikasi):** cmake menemukan TensorRT (`/usr/lib/aarch64-linux-gnu/libnvinfer.so`), CUDA 12.6; `make` mencapai 100% dan menghasilkan `llm_build`, `llm_inference`, `llm_stream`, `llm_bench`, dan **`build/libNvInfer_edgellm_plugin.so`** (plugin atensi kustom).

## Part C — Build engine TensorRT (ONNX → `.engine`)

`llm_build` mem-parse ONNX dan membangun engine TensorRT yang dioptimalkan untuk GPU Orin.

### Gotcha #5 — plugin path
ONNX memakai custom op `trt.attention_plugin` → TensorRT harus me-load **`libNvInfer_edgellm_plugin.so`**. Set `EDGELLM_PLUGIN_PATH` ke path **absolut**-nya (untuk build *dan* inference).

### Gotcha #6 — OOM saat autotuning (kunci untuk Orin Nano 8 GB)
Build engine default (`--maxInputLen 1024 --maxKVCacheCapacity 4096`) **OOM** di memori *unified* 8 GB (gejala: `NvMapMemAllocInternalTagged error 12`, `CUDA error 2`). **Fix (sesuai troubleshooting NVIDIA):** turunkan profil ke **`--maxInputLen 256 --maxKVCacheCapacity 512`** dan kosongkan cache: `sudo sysctl -w vm.drop_caches=3`. `llm_build` otomatis menyalin engine + sidecar (config, tokenizer, chat-template, embedding) ke `engineDir`.

In [ ]:
# === Part C: build engine TensorRT ===  (di Orin)
export PATH=/usr/local/cuda/bin:$PATH
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib/aarch64-linux-gnu:$LD_LIBRARY_PATH
export EDGELLM_PLUGIN_PATH=/media/jetson/DATA/edgellm-work/TensorRT-Edge-LLM/build/libNvInfer_edgellm_plugin.so
cd /media/jetson/DATA/edgellm-work
R=TensorRT-Edge-LLM/build/examples/llm

sudo sysctl -w vm.drop_caches=3      # bebaskan page cache dulu (Orin Nano 8 GB)

$R/llm_build \
  --onnxDir ./qwen3-onnx/llm \
  --engineDir ./qwen3-engines \
  --maxBatchSize 1 \
  --maxInputLen 256 \
  --maxKVCacheCapacity 512           # profil hemat memori untuk Orin Nano 8 GB

ls -lh ./qwen3-engines               # llm.engine (~1.2 GB) + config.json + embedding + tokenizer

**Output (terverifikasi):** `AttentionPlugin` ter-load, `Engine generation completed in ~196 s`, peak GPU **1136 MiB**, lalu `LLM engine built successfully`. `engineDir` berisi `llm.engine` (1.2 GB) + sidecar lengkap.

## Part D — Inference + benchmark

`llm_inference` me-load engine + sidecar, menerapkan chat-template, dan menjalankan permintaan dari file JSON.

In [ ]:
# === Part D: inference (spesialis ekstraktor JSON) ===  (di Orin)
export LD_LIBRARY_PATH=/usr/local/cuda/lib64:/usr/lib/aarch64-linux-gnu:$LD_LIBRARY_PATH
export EDGELLM_PLUGIN_PATH=/media/jetson/DATA/edgellm-work/TensorRT-Edge-LLM/build/libNvInfer_edgellm_plugin.so
cd /media/jetson/DATA/edgellm-work
R=TensorRT-Edge-LLM/build/examples/llm

cat > ./input.json << 'EOF'
{"batch_size":1,"temperature":0.0,"top_p":1.0,"top_k":1,"max_generate_length":96,
 "requests":[{"messages":[
   {"role":"system","content":"Ekstrak pesanan ke JSON dengan key: produk, jumlah, harga_satuan, total. Angka tanpa titik. Keluarkan HANYA JSON."},
   {"role":"user","content":"Pesan 3 kopi, harga 25.000 per item."}
 ]}]}
EOF

$R/llm_inference --engineDir ./qwen3-engines --inputFile ./input.json --outputFile ./output.json
cat ./output.json

# Benchmark throughput decode di Orin:
$R/llm_bench --engineDir ./qwen3-engines --mode decode --osl 128 --pastKVLen 128

**Output (terverifikasi di Orin Nano):**
```json
"output_text": "{\"harga_satuan\": 25000, \"jumlah\": 3, \"produk\": \"kopi\", \"total\": 75000}",
"finish_reason": "end-of-sequence"
```
Spesialis kita **berjalan offline di Orin** dan mengekstrak pesanan → JSON dengan **benar** (3 × 25.000 = 75.000), tanpa internet, tanpa cloud, tanpa Python di jalur inference.

**Benchmark decode:** `Per-step avg: 23.44 ms` → **≈ 42.7 tokens/sec** (FP16, batch 1, Orin Nano 8 GB). Lebih dari cukup untuk respons interaktif di edge.

## "Worth the effort?" — TensorRT Edge-LLM vs Ollama (jujur, dengan angka)

Spesialis yang sama juga sudah kita deploy di Modul 04 lewat **Ollama** (GGUF). Mari bandingkan **di Orin Nano yang sama, model & prompt yang sama** — karena kerja keras hanya layak jika ada bayarannya.

| Aspek | **TensorRT Edge-LLM** (FP16) | **Ollama** (GGUF Q8_0) |
|-------|------------------------------|------------------------|
| **tokens/sec** (Orin Nano 8 GB) | **42.7** | **40.4** |
| Output (prompt sama) | `{"harga_satuan":25000,"jumlah":3,"produk":"kopi","total":75000}` | **identik & benar** |
| Effort setup | ~1 jam, **6 gotcha** (venv export + cmake/make ~20 mnt + plugin path + engine OOM-tuning) | **1 perintah**: `ollama run hf.co/chmdznr/qwen3-0.6b-spesialis-gguf` |
| Webservice OpenAI `/v1` | ❌ CLI (perlu dibungkus FastAPI/vLLM) | ✅ **bawaan** |
| Footprint | engine 1.2 GB (FP16) | 639 MB (Q8_0) |

**Verdict untuk kasus ini (spesialis 0.6B, Orin Nano):** **Ollama menang telak.** Kecepatan **praktis sama** (selisih ~5%, dalam noise), output identik, tapi setup **jauh lebih sederhana** dan langsung memberi webservice OpenAI. Untuk belajar, model kecil, dan serving cepat di edge — **pakai Ollama.**

**Lalu kapan TensorRT Edge-LLM baru sepadan?** Kelebihannya **tidak terlihat** pada microbench model 0.6B. Ia mulai menang ketika:
- **Model lebih besar / latency super-ketat** — kernel fusion + utilisasi Tensor Core + paged-KV TensorRT-LLM unggul saat compute jadi bottleneck (bukan kasus 0.6B).
- **Integrasi produksi C++** (robot, kamera, kiosk) — runtime **tanpa interpreter Python** di jalur inference: overhead lebih rendah, footprint & latency lebih deterministik.
- **Quantization NVFP4/FP8** pada Tensor Core, dan kontrol penuh atas optimasi engine.

**Pelajaran rekayasa (penting untuk sertifikasi):** "NVIDIA-native" ≠ otomatis "lebih baik". **Ukur dulu.** Untuk banyak skenario edge praktis, jalur GGUF/Ollama sudah memberi 95% manfaat dengan 5% usaha. TensorRT Edge-LLM adalah palu yang tepat untuk paku produksi-skala-besar — bukan untuk setiap paku.

## Yang kita pelajari di nb06

| Tahap | Perintah inti | Catatan edge |
|-------|---------------|--------------|
| Export ONNX | `tensorrt-edgellm-export <model> <out>` | **portabel** (CPU venv torch 2.12; bisa di Colab/x86) |
| Build runtime | `cmake -DEMBEDDED_TARGET=jetson-orin … && make -j4` | native `aarch64`+`sm_87`; **env CUDA wajib** |
| Build engine | `llm_build --onnxDir … --maxInputLen 256 --maxKVCacheCapacity 512` | **terikat hardware**; `EDGELLM_PLUGIN_PATH`; 256/512 + `drop_caches` agar muat 8 GB |
| Inference | `llm_inference --engineDir … --inputFile …` | C++ murni, offline, 42.7 tok/dtk |

**Kumpulan gotcha (peta jalan jika kamu ulang di Orin lain):**
1. `huggingface_hub < 1.0` (kompatibilitas `transformers` 4.57).
2. torch sistem Jetson terlalu lama → **venv `torch==2.12.0+cpu` terisolasi** (bukan container ~20 GB; root-disk kecil + Docker 29 containerd di root).
3. **Export env CUDA** sebelum cmake (`nvcc` perlu PATH/`CUDA_HOME`).
4. `-DEMBEDDED_TARGET=jetson-orin` (default arch repo bukan `sm_87`); `make -j4` (RAM 8 GB).
5. `EDGELLM_PLUGIN_PATH` (absolut) untuk build & inference.
6. OOM engine → `--maxInputLen 256 --maxKVCacheCapacity 512` + `sudo sysctl -w vm.drop_caches=3`.

**Pelajaran besar (untuk sertifikasi):** model yang sama dapat di-*serve* melalui banyak backend lewat kontrak yang sama — **NIM hosted** (nb02), **NIM self-host RTX**, dan **TensorRT Edge-LLM di Jetson** (nb06). Yang **portabel** adalah ONNX; **engine TensorRT terikat ke GPU + versi**, sehingga build engine & inference selalu di perangkat target. Inilah inti *edge deployment*: model jalan **di tempat data berada**, offline dan latency rendah.

> 🔗 Menutup Modul 06: dari **GPU & optimasi** (nb01) → **serving Triton/Dynamo/NIM** (nb02) → **Trustworthy AI & guardrails** (nb03–04) → **capstone `/ask` berpagar** (nb05) → **deploy NVIDIA-native di edge** (nb06 ini).

### Latihan

1. **Ganti precision.** Edge-LLM mendukung quantization **INT4 AWQ** (lebih kecil dari FP16). Jalankan `tensorrt-edgellm-quantize` lebih dulu (butuh `nvidia-modelopt`), lalu export+build dari checkpoint terkuantisasi. Bandingkan ukuran `llm.engine` dan tokens/sec vs FP16 di sini.
2. **Naikkan profil.** Bangun ulang engine dengan `--maxInputLen 512 --maxKVCacheCapacity 1024`. Apakah masih muat di 8 GB? Pakai `tegrastats` untuk memantau RAM saat build, dan catat di titik mana OOM muncul.
3. **Webservice di edge.** `llm_inference` adalah CLI, bukan server OpenAI. Bungkus engine dengan **FastAPI `/ask`** tipis (ingat nb05) atau pakai **vLLM-on-Jetson** agar Orin meng-expose `/v1/chat/completions` — sehingga kode klien dari nb02 berjalan tanpa perubahan, kini menunjuk ke Orin-mu.
4. **Bandingkan jalur.** Jalankan model GGUF yang sama lewat **Ollama** di Orin (`ollama run hf.co/chmdznr/qwen3-0.6b-spesialis-gguf`) dan bandingkan tokens/sec + kemudahan setup vs jalur TensorRT Edge-LLM ini. Kapan masing-masing lebih tepat?